# Employee and Manager Analysis with SQL

This notebook demonstrates core SQL skills using a small employee-management database. It creates two relational tables, loads sample records, and answers common workforce-analysis questions involving filtering, pattern matching, joins, calculated fields, and subqueries.

## Skills demonstrated

- Creating and populating SQLite tables
- Defining primary keys and relationships
- Filtering rows with `WHERE`
- Matching text with `LIKE`
- Joining related tables
- Calculating derived values
- Using nested subqueries
- Writing readable, well-structured SQL


## 1. Environment setup


In [ ]:
!pip install -q prettytable==3.10.0 ipython-sql==0.5.0

%load_ext sql
%sql sqlite://
%config SqlMagic.style = "DEFAULT"


## 2. Database design

The database contains:

- `employees`: employee number, name, salary, department, and age
- `managers`: manager number, name, salary, department, and age

Each department has one manager, linked through `dept_id`.


In [ ]:
%%sql

DROP TABLE IF EXISTS employees;
DROP TABLE IF EXISTS managers;

CREATE TABLE employees (
    employee_num INTEGER PRIMARY KEY,
    last_name TEXT NOT NULL,
    first_name TEXT NOT NULL,
    salary INTEGER NOT NULL CHECK (salary >= 0),
    dept_id INTEGER NOT NULL,
    age INTEGER NOT NULL CHECK (age > 0)
);

CREATE TABLE managers (
    mgr_num INTEGER PRIMARY KEY,
    last_name TEXT NOT NULL,
    first_name TEXT NOT NULL,
    salary INTEGER NOT NULL CHECK (salary >= 0),
    dept_id INTEGER NOT NULL UNIQUE,
    age INTEGER NOT NULL CHECK (age > 0)
);


## 3. Load sample data


In [ ]:
%%sql

INSERT INTO employees VALUES
    (1001, 'Smith', 'John', 62000, 500, 32),
    (1002, 'Anderson', 'Jane', 87500, 500, 40),
    (1003, 'Everest', 'Brad', 71000, 501, 26),
    (1004, 'Horvath', 'Brian', 62000, 501, 44);

INSERT INTO managers VALUES
    (101, 'White', 'Jonathan', 152000, 500, 38),
    (102, 'Smith', 'Paul', 120000, 501, 40),
    (103, 'Cowgill', 'Richard', 150000, 502, 50);


### Verify the tables


In [ ]:
%%sql

SELECT * FROM employees;


In [ ]:
%%sql

SELECT * FROM managers;


## 4. Employees older than 30


In [ ]:
%%sql

SELECT
    last_name,
    first_name,
    age
FROM employees
WHERE age > 30
ORDER BY age DESC;


**Interpretation:** This query identifies employees above age 30 and orders them from oldest to youngest.


## 5. Employees whose first names end in `n`


In [ ]:
%%sql

SELECT
    last_name,
    first_name
FROM employees
WHERE first_name LIKE '%n'
ORDER BY last_name;


**Interpretation:** The `%` wildcard matches any number of preceding characters, while `n` requires the name to end with that letter.


## 6. Managers earning between $130,000 and $160,000


In [ ]:
%%sql

SELECT
    last_name,
    first_name,
    salary
FROM managers
WHERE salary > 130000
  AND salary < 160000
ORDER BY salary DESC;


## 7. Employees older than their managers


In [ ]:
%%sql

SELECT
    e.last_name AS employee_last_name,
    e.first_name AS employee_first_name,
    m.last_name AS manager_last_name,
    m.first_name AS manager_first_name,
    e.age - m.age AS age_difference
FROM employees AS e
INNER JOIN managers AS m
    ON e.dept_id = m.dept_id
WHERE e.age > m.age
ORDER BY age_difference DESC;


**Interpretation:** The join connects employees to the manager responsible for the same department. The calculated column shows the age difference.


## 8. Employee or employees with the second-highest distinct salary


In [ ]:
%%sql

SELECT
    employee_num,
    last_name,
    first_name,
    salary,
    dept_id,
    age
FROM employees
WHERE salary = (
    SELECT MAX(salary)
    FROM employees
    WHERE salary < (
        SELECT MAX(salary)
        FROM employees
    )
);


## Conclusion

This project demonstrates practical SQL techniques used in data-analysis work: table creation, data validation, filtering, wildcard searches, joins, calculated fields, ordering, and nested subqueries. The database is intentionally small so each query and result can be inspected easily.
